# Task 1: Traffic Speed Prediction — XGBoost Submission

Predict traffic speed (km/h) for 1,260 road segments at +20, +40, and +60 minutes into the future.

**Approach**: Global XGBoost model with handcrafted features from speed history, road network adjacency, and road metadata.

In [ ]:
import csv
import json
import pickle
from pathlib import Path
from datetime import datetime

import numpy as np
import xgboost as xgb

DATASET_DIR = Path("dataset-task1")
WINDOW = 15
HORIZONS = [5, 10, 15]  # steps = +20, +40, +60 min
N_ROADS = 1260

## 1. Data Loading

Load training speed series (two blocks, 16,199 total timesteps), event text per timestep, road network adjacency, and road metadata.

In [ ]:
def load_train_speeds():
    s1 = np.load(DATASET_DIR / "train" / "train_speed_m1_1_11160.npy")
    s2 = np.load(DATASET_DIR / "train" / "train_speed_m2_1_5039.npy")
    return s1, s2


def load_train_texts():
    with open(DATASET_DIR / "train" / "train_text_m1_1_11160.json") as f:
        t1 = json.load(f)
    with open(DATASET_DIR / "train" / "train_text_m2_1_5039.json") as f:
        t2 = json.load(f)

    def to_sorted_list(text_dict):
        keys = sorted(text_dict.keys(), key=lambda k: int(k.split("_")[1]))
        return [text_dict[k] for k in keys]

    return to_sorted_list(t1), to_sorted_list(t2)


def load_test_data():
    hist = np.load(DATASET_DIR / "test" / "test_X_hist.npy")
    with open(DATASET_DIR / "test" / "test_texts.json") as f:
        texts_dict = json.load(f)
    keys = sorted(texts_dict.keys(), key=lambda k: int(k.split("_")[1]))
    return hist, [texts_dict[k] for k in keys]


def load_adjacency():
    return np.load(DATASET_DIR / "static" / "matrix.npy")  # (1260, 1260) int8


def load_roads():
    with open(DATASET_DIR / "static" / "Roads1260.json") as f:
        return json.load(f)

## 2. Supervised Windows

Slide a 15-step window across the speed series to create supervised samples.
Each sample: 15-step history → targets at +5, +10, +15 steps ahead (corresponding to +20, +40, +60 minutes).

In [ ]:
def build_windows(speeds, texts, stride=1):
    T = len(speeds)
    max_horizon = max(HORIZONS)
    max_t = T - WINDOW - max_horizon

    X, Y = [], []
    for t in range(0, max_t, stride):
        X.append(speeds[t : t + WINDOW])
        y = np.stack([speeds[t + WINDOW + h] for h in HORIZONS], axis=0)
        Y.append(y)

    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)


s1, s2 = load_train_speeds()
t1, t2 = load_train_texts()
adj = load_adjacency()
roads = load_roads()

X1, Y1 = build_windows(s1, t1)
X2, Y2 = build_windows(s2, t2)

X_all = np.concatenate([X1, X2], axis=0).astype(np.float32)
Y_all = np.concatenate([Y1, Y2], axis=0).astype(np.float32)

print(f"Train samples: {len(X_all)}")

## 3. Feature Engineering

For each road in each window, extract 12 features in a vectorized pass (no Python loops over roads):

- **Lag speeds** at t−0, t−1, t−3, t−7, and the earliest step (captures recent trends and seasonality)
- **Statistics**: mean, standard deviation, and trend (last − first) over the 15-step window
- **Road metadata**: road class and segment length (from Roads1260.json)
- **Neighbor speeds**: average speed of adjacent roads (from the adjacency matrix) at the most recent step and 3 steps back — this encodes spatial context

In [ ]:
def build_features(hist_windows, adj, roads):
    N, T, R = hist_windows.shape

    roadclass = np.array([roads[r][0].get("roadclass", 0) for r in range(R)], dtype=np.float32)
    length = np.array([roads[r][0].get("length", 0) for r in range(R)], dtype=np.float32)

    lags = np.stack(
        [hist_windows[:, -1, :],
         hist_windows[:, -2, :],
         hist_windows[:, -4, :],
         hist_windows[:, -8, :],
         hist_windows[:, 0, :]],
        axis=-1,
    )

    mean_h = hist_windows.mean(axis=1)
    std_h = hist_windows.std(axis=1)
    trend = hist_windows[:, -1, :] - hist_windows[:, 0, :]

    degrees = adj.sum(axis=1, keepdims=True).clip(min=1)
    adj_norm = adj.astype(np.float32) / degrees.astype(np.float32)

    neighbor_last = hist_windows[:, -1, :] @ adj_norm.T
    neighbor_3 = hist_windows[:, -3, :] @ adj_norm.T

    feats = np.stack(
        [lags[:, :, 0], lags[:, :, 1], lags[:, :, 2], lags[:, :, 3], lags[:, :, 4],
         mean_h, std_h, trend,
         np.broadcast_to(roadclass, (N, R)),
         np.broadcast_to(length, (N, R)),
         neighbor_last, neighbor_3],
        axis=-1,
    )

    return feats.reshape(-1, 12).astype(np.float32)


print("Building features...")
F_all = build_features(X_all, adj, roads)
Y_flat = Y_all.transpose(0, 2, 1).reshape(-1, 3)
print(f"Features: {F_all.shape}, Targets: {Y_flat.shape}")

## 4. Train XGBoost Models

Train one XGBoost regressor per horizon (h5, h10, h15). Each model maps the 12 engineered features to the future speed at that horizon. A single global model is shared across all 1,260 roads — road identity is encoded through metadata and neighbor features, not one-hot road IDs.

The test metric is Mean Squared Error (MSE), so each regressor minimizes squared error directly.

In [ ]:
models = {}
for hi, hname in enumerate(["h5", "h10", "h15"]):
    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, verbosity=0, n_jobs=-1,
    )
    model.fit(F_all, Y_flat[:, hi])
    models[hname] = model
    print(f"  {hname} done")

## 5. Predict on Test Set

The test set contains 540 fixed history windows with no targets. Apply the same feature pipeline, predict all three horizons for every road, and clip predictions to ≥ 0 (speed cannot be negative).

In [ ]:
test_hist, _ = load_test_data()
F_test = build_features(test_hist, adj, roads)
print(f"Test features: {F_test.shape}")

test_preds = np.zeros((540, 3, N_ROADS), dtype=np.float32)
for hi, hname in enumerate(["h5", "h10", "h15"]):
    test_preds[:, hi, :] = models[hname].predict(F_test).reshape(540, N_ROADS)
test_preds = test_preds.clip(min=0)

print(f"Predictions: {test_preds.shape}, range=[{test_preds.min():.1f}, {test_preds.max():.1f}]")

## 6. Write & Validate Submission

Generate `submission.csv` with the exact format:

```
id,speed
test_00000_h5_r0,44.0
test_00000_h5_r1,73.0
...
```

Each id encodes `test_{sample:05d}_h{horizon}_r{road}`. The file is validated against `sample_submission.csv` to ensure all 2,041,200 row IDs match exactly in count and order.

In [ ]:
def write_submission(predictions, label="", models_dict=None):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    name = f"{ts}_{label}" if label else ts
    out_dir = Path("submissions") / name
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / "submission.csv"
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "speed"])
        for s in range(540):
            for hi, hn in enumerate(["h5", "h10", "h15"]):
                for r in range(N_ROADS):
                    writer.writerow([f"test_{s:05d}_{hn}_r{r}", f"{predictions[s, hi, r]:.6f}"])

    expected = 540 * 3 * N_ROADS + 1
    with open(out_path) as f:
        actual = sum(1 for _ in f)
    assert actual == expected, f"Row count mismatch: {actual} vs {expected}"

    if models_dict:
        for mname, m in models_dict.items():
            with open(out_dir / f"model_{mname}.pkl", "wb") as f:
                pickle.dump(m, f)

    print(f"Saved to {out_dir}/ ({actual - 1} predictions)")
    return out_dir


def validate_submission(submission_path):
    sample_path = DATASET_DIR / "sample_submission.csv"
    with open(sample_path) as f:
        sample_ids = [row[0] for row in csv.reader(f)][1:]
    with open(submission_path) as f:
        sub_ids = [row[0] for row in csv.reader(f)][1:]

    if len(sub_ids) != len(sample_ids):
        raise ValueError(f"Row count mismatch: {len(sub_ids)} vs {len(sample_ids)}")

    mismatches = [(i, s, g) for i, (s, g) in enumerate(zip(sub_ids, sample_ids)) if s != g]
    if mismatches:
        for i, s, g in mismatches[:5]:
            print(f"  Row {i + 2}: got {s}, expected {g}")
        raise ValueError(f"{len(mismatches)} id mismatches")

    print(f"Validated: {len(sub_ids)} rows, all ids match sample_submission.csv")


out_dir = write_submission(test_preds, label="xgb_final", models_dict=models)
validate_submission(out_dir / "submission.csv")